# LEVELS Benchmark — Level 2: Workflow

Multi-step pipelines combining ≥2 data-management categories (e.g. search → download →
summarize, filter → report, ingest → validate). Run **non-isolated** (see `run_level`), so
tasks sharing a list may see each other's conversational context; tasks that need guaranteed
clean state use `setup_query` or an explicit `steps` list of turns run against one shared
explorer (`task.steps`, see `run_task`). Scored by `score_level2()`: every tool in
`expected_tools` must be used at least once (not any-of, like Level 1), plus an LLM-judged
process-quality score against `success_criteria`. Ground-truth counts/fields below were
hand-verified against `data/oceans_11/*.db` directly with sqlite3, the same way Level 1's
were verified via the DSI CLI.

#### How `score_level2()` scores a task

| Component | What it checks | Pass condition | Contributes to |
|---|---|---|---|
| **Tool coverage** (`tools_ok`) | Did the agent call *every* tool listed in `expected_tools`? | ALL required tools must appear in `tool_calls` (unlike Level 1, where only *one* correct tool is needed) | Correctness |
| **Quantitative check** (`quant_ok`) | Does the expected number (`expected_count`) show up in the tool results or final response? | Regex match for the number as a `count:`/`missing_count:` field or a standalone number in the combined text blob | Correctness |
| **Keyword check** (`keyword_ok`) | Do all required facts (`response_keywords`) appear in the response/tool output? | ALL keywords must be present (case-insensitive) — every one, not just any one | Correctness |
| **Forbidden-keyword check** (`forbidden_ok`) | Did the agent avoid stating any wrong/distractor facts (`forbidden_keywords`)? | None of the forbidden terms may appear | Correctness |
| **Non-empty check** (`nonempty`) | Did the agent actually produce a final answer? | `response` must be non-blank | Correctness |
| **Correctness** (`result.passed`) | All five checks above | `tools_ok AND quant_ok AND keyword_ok AND forbidden_ok AND nonempty` | Binary 0.0 / 1.0 — **50% of final score** |
| **Process quality** (`judge_scores["process_quality_score"]`) | Was the *method* good — efficient, well-ordered, matching the `success_criteria` rubric — regardless of whether the final answer was right? | LLM judge (both models, scores averaged) rates 0.0 (ignored rubric / haphazard) to 1.0 (clean, sound multi-step approach) | Continuous 0.0–1.0 — **50% of final score** |

**Final score formula:**
- If the task has `success_criteria` set: `score = 0.5 × correctness (0 or 1) + 0.5 × process_quality_score`
- If no `success_criteria`: `score = correctness` only (0.0 or 1.0)

**Important nuance:** `result.passed` reflects **correctness only** — a task can "pass" (did all the right steps, got the right facts) while still scoring well below 1.0 overall if the LLM judge dings it for a messy/inefficient process. That's deliberate: it lets the benchmark distinguish "got it right, cleanly" from "got it right, but wastefully."

**Session isolation:** every task runs in a fresh agent session. Tasks needing multi-turn context already define their own `steps`, so isolation is what allows them to run in parallel. To reproduce the original single shared session per model, pass `share_session=True` to `run_benchmark`. Only models then run in parallel.

Shared harness code (scoring, parallel runner, reports) lives in `levels_bench/`, and the task
definitions in `levels_bench/tasks/`. Each (model, task) pair runs as an independent job in a
pool of worker processes, so `max_workers` runs proceed at once. Every finished result is
appended to `levels_benchmark_results/<run_id>_<label>.partial.jsonl` as it completes. If a run
is interrupted, call `run_benchmark(..., resume=True)` to run only what's missing.

## Setup

In [ ]:
# Run once per environment (not needed on every run):
# %pip install -q -r requirements.txt
# %pip install -q --no-build-isolation -e ../..

In [ ]:
from levels_bench import init, run_benchmark, flatten, results_to_dataframe, load_results, reports, debug_task
from levels_bench.tasks import LEVEL2_TASKS, get_task

cfg = init(
    # models_to_run=["aws-gov.gpt-5.4"],   # default: every model in levels_bench.config.POSSIBLE_MODELS
    # judge_models=[...],                   # default: same list; judged scores are averaged across them
    max_workers=4,                          # concurrent agent runs; raise if the portal's rate limit allows
    task_timeout_sec=180,                   # per agent turn
    # skip_network=True,                    # skip tasks that download from oceans11.lanl.gov
)

In [ ]:
reports.task_table(LEVEL2_TASKS)

## Run

In [ ]:
l2_results = flatten(run_benchmark(LEVEL2_TASKS, "level2_workflow"))
# To analyze an earlier run without re-running it:
# l2_results = load_results("level2_workflow")
l2_df = results_to_dataframe(l2_results)
l2_df

## Results

In [ ]:
reports.model_summary(l2_df, extra_judge_cols=['judge_process_quality_score'])

In [ ]:
reports.failures(l2_df)

In [ ]:
reports.timing_breakdown(l2_results)

## Inspect / debug

Transcripts are recorded for every run. `debug_task` reruns one task in this process with live output.

In [ ]:
# reports.transcript(l2_results, "L2_physics_contact_report")
# tr = debug_task(get_task("L2_physics_contact_report"), model="anthropic.claude-sonnet-4-5-20250929-v1:0")